# Cellpose on Green: zero-shot, fine-tuning and predictions (Kaggle)

Cellpose is a segmentation tool pre-trained on many images of cells; here its version 4 (Cellpose-SAM, ~290 M parameters)
is compared with our pipeline. In one session on a Kaggle GPU this notebook:

1. **zero-shot:** segments the training, validation and test images with the pre-trained model, which has never seen our labels;
2. **fine-tuning:** trains the same model on our 169 training images;
3. **predictions of the fine-tuned model** on the training, validation and test images: masks and cell probabilities
   (the object classifier of "Cellpose fine-tuned + classifier", notebook 04, is trained on the training images and uses
   the probabilities).

Output: `results_cellpose.zip`, with one file per split for the zero-shot model (`cellpose_<split>.npz`: masks) and for the
fine-tuned model (`cellpose_ftp_<split>.npz`: masks and probabilities), and the settings (`cellpose_config.json`). The model
weights are not included. The test images are evaluated only in `notebooks/04_test.ipynb`.

Before running:
1. **Settings → Accelerator → GPU T4 x2** and **Settings → Internet → On** (to install Cellpose and download its weights;
   it needs a phone-verified Kaggle account).
2. **Add Input**: the datasets `cellseg-code` (for the split) and `fnc-green-data` (images and cleaned masks), described in
   `kaggle/README.md`.
3. Short check: `PROBE = True` (section 2) → **Run All**: 2 images per split, 2 fine-tuning epochs on 8 images (~10 min).
4. Full run: `PROBE = False` → **Save Version → Save & Run All (Commit)** (about 7 h).

## 1. Install Cellpose

The version used for the results of this repository. NumPy and SciPy are kept at the versions already loaded by the
kernel: changing them in the middle of a session breaks the later imports.

In [ ]:
import numpy
import scipy
!pip install -q "cellpose==4.2.1.1" "numpy=={numpy.__version__}" "scipy=={scipy.__version__}"

from importlib.metadata import version
print("cellpose", version("cellpose"))

## 2. Data

The same images and the same train / validation split as our network. Cellpose receives only the green channel, which
carries the c-FOS signal.

In [ ]:
import json
import shutil
import time
from pathlib import Path

import numpy as np
import skimage.io as io
import torch
from skimage import measure

PROBE = True        # True: short check (2 images per split, 2 fine-tuning epochs); False: full run

assert torch.cuda.is_available(), "no GPU: Settings → Accelerator → GPU T4 x2"
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/runs/cellpose_probe" if PROBE else "/kaggle/working/runs/cellpose")
OUT.mkdir(parents=True, exist_ok=True)

def find_one(pattern):
    '''First path under /kaggle/input matching the pattern (Kaggle may nest the dataset folders).'''
    hits = sorted(INPUT.rglob(pattern))
    assert hits, f"not found under {INPUT}: {pattern} (were both datasets added with 'Add Input'?)"
    return hits[0]

split = json.load(open(find_one("splits/green_split.json")))
TRAINVAL = find_one("green/trainval/images")
TEST = find_one("green/test/images")
MASKS = find_one("cleaned_masks/trainval/Green/masks")
SPLITS = {"train": (TRAINVAL, split["train"]),
          "val": (TRAINVAL, split["val"]),
          "test": (TEST, sorted(p.name for p in TEST.glob("*.png")))}
assert [len(names) for _, names in SPLITS.values()] == [169, 43, 70]
if PROBE:
    SPLITS = {s: (folder, names[:2]) for s, (folder, names) in SPLITS.items()}

def signal(folder, name):
    '''The green channel (the c-FOS signal) as a 2-D image: the input of Cellpose.'''
    return io.imread(folder / name)[..., 1]

## 3. Zero-shot

The pre-trained model as downloaded, without training on our labels. Cell diameter 30 px (the mean diameter of the
training cells is 30.6 px); the other settings are the defaults of Cellpose. `segment` returns, for every image, the
masks (one integer per cell, 0 = background) and the cell probability: the sigmoid of Cellpose's cell logit
(`cellprob`), stored as an integer 0–255.

In [ ]:
from cellpose import models, train

DIAMETER = 30

def segment(model, split_name):
    '''Masks and cell probabilities (× 255) of every image of a split.'''
    folder, names = SPLITS[split_name]
    masks, probs = [], []
    for name in names:
        image = signal(folder, name)
        mask, flows = model.eval(image, diameter=DIAMETER)[:2]
        cellprob = np.clip(flows[2], -30, 30)          # the cell logit; beyond ±30 the sigmoid is already 0 or 1
        assert cellprob.shape == image.shape
        masks.append(mask.astype(np.uint16))
        probs.append(np.round(255 / (1 + np.exp(-cellprob))).astype(np.uint8))
    return np.stack(masks), np.stack(probs)

zero_shot = models.CellposeModel(gpu=True)
for s, (folder, names) in SPLITS.items():
    start = time.time()
    masks, _ = segment(zero_shot, s)
    np.savez_compressed(OUT / f"cellpose_{s}.npz", masks=masks, names=np.array(names))
    print(f"zero-shot, {s}: {len(names)} images, {(time.time() - start) / len(names):.1f} s per image")

del zero_shot                 # frees the GPU memory before the training
torch.cuda.empty_cache()

## 4. Fine-tuning

The same pre-trained model, trained on the 169 training images with our labels (one integer per annotated cell, from the
cleaned masks). The validation images are never used: the number of epochs (100) is fixed in advance, and the other
settings are the defaults of Cellpose's `train_seg`. One exception: `min_train_masks=0`. By default Cellpose skips the
images with fewer than 5 cells (29 of the 169), which are exactly those where the annotators left out the faint cells;
with 0 it trains on the same 169 images as our network.

In [ ]:
N_EPOCHS = 2 if PROBE else 100
train_names = split["train"]
if PROBE:   # 6 images + the training images without cells (these must not break the training)
    train_names = train_names[:6] + [n for n in train_names if io.imread(MASKS / n).max() == 0]
train_images = [signal(TRAINVAL, n) for n in train_names]
train_labels = [measure.label(io.imread(MASKS / n) > 0, connectivity=1).astype(np.int32) for n in train_names]

MODEL_DIR = Path("/kaggle/working/cellpose_model")      # the weights, outside the results zip
MODEL_DIR.mkdir(exist_ok=True)                           # train_seg only creates the "models" subfolder
model = models.CellposeModel(gpu=True)
start = time.time()
# the warnings "Mean of empty slice" come from the images without cells: harmless
model_path, losses = train.train_seg(model.net, train_data=train_images, train_labels=train_labels, n_epochs=N_EPOCHS,
                                     min_train_masks=0, save_path=str(MODEL_DIR), model_name="cellpose_ft_green")[:2]
assert np.isfinite(losses).all(), "NaN or inf in the training loss"
print(f"fine-tuning: {len(train_names)} images, {N_EPOCHS} epochs, {(time.time() - start) / 60:.0f} min | "
      f"loss {losses[0]:.3f} → {losses[-1]:.3f}")

## 5. Predictions of the fine-tuned model

Masks and cell probabilities on the training, validation and test images, with the same settings as the zero-shot model.

In [ ]:
model = models.CellposeModel(gpu=True, pretrained_model=str(model_path))
for s, (folder, names) in SPLITS.items():
    start = time.time()
    masks, probs = segment(model, s)
    np.savez_compressed(OUT / f"cellpose_ftp_{s}.npz", masks=masks, probs=probs, names=np.array(names))
    print(f"fine-tuned, {s}: {len(names)} images, {(time.time() - start) / len(names):.1f} s per image")

## 6. Settings and results

The settings in `cellpose_config.json`, then one zip with the predictions and the settings (no weights): download it from
the **Output** tab of the version.

In [ ]:
config = {"cellpose_version": version("cellpose"), "model": "Cellpose-SAM (default model of the version)",
          "input": "green channel", "diameter": DIAMETER, "other_settings": "Cellpose defaults",
          "fine_tuning": {"train_images": len(train_names), "n_epochs": N_EPOCHS, "min_train_masks": 0},
          "probability": "sigmoid(cellprob) x 255, uint8", "probe": PROBE}
with open(OUT / "cellpose_config.json", "w") as f:
    json.dump(config, f, indent=2)

shutil.make_archive("/kaggle/working/results_cellpose", "zip", root_dir="/kaggle/working", base_dir=f"runs/{OUT.name}")
print(sorted(p.name for p in OUT.iterdir()))